In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

print("✓ Libraries imported successfully")

# COMMAND ----------

# DBTITLE 1,Step 2: Load Source Tables
# MAGIC %md
# MAGIC ## Step 2: Load Source Tables
# MAGIC
# MAGIC Load both source tables from Unity Catalog and examine their schemas.

# COMMAND ----------

# DBTITLE 1,Load products table
# Load the products table
products_df = spark.table("enterprise_ai.raw_data.product_docs")

print("Products Table Schema:")
products_df.printSchema()
print(f"\nTotal products: {products_df.count()}")

# Display sample data
print("\nSample products:")
display(products_df.limit(5))


In [0]:
# =========================
# READ TABLES
# =========================

profit_df = spark.table(
    "enterprise_ai.raw_data.profit_loss"
)
display(profit_df.limit(5))
# DBTITLE 1,Load balance sheet table
# Load the balance sheet table


balance_df = spark.table(
    "enterprise_ai.raw_data.balance_sheet"
)
display(balance_df.limit(5))

# DBTITLE 1,Load employee table
#
employee_df = spark.table(
    "enterprise_ai.raw_data.employee_data"
)
display(employee_df.limit(5))

# DBTITLE 1,Load document
doc_df = spark.table(
    "enterprise_ai.raw_data.product_docs"
)
display(doc_df.limit(5))

# DBTITLE 1,Step 3: Join Tables
# MAGIC %md
# MAGIC ## Step 3: Join Tables
# MAGIC
# MAGIC Join the three tables together using the `company`
# =========================
# CLEAN COLUMN NAMES
# =========================

def clean_columns(df, prefix):

    new_cols = []

    for c in df.columns:

        if c in [
            "company",
            "financial_year"
            
        ]:

            new_cols.append(c)

        else:

            clean_name = (
                c.strip()
                 .lower()
                 .replace(" ", "_")
                 .replace("-", "_")
                 .replace("(", "")
                 .replace(")", "")
            )

            new_cols.append(
                f"{prefix}_{clean_name}"
            )

    return df.toDF(*new_cols)

# =========================
# APPLY CLEANING
# =========================

profit_df = clean_columns(
    profit_df,
    "profit"
)

balance_df = clean_columns(
    balance_df,
    "balance"
)

employee_df = clean_columns(
    employee_df,
    "employee"
)

# =========================
# JOIN STRUCTURED TABLES
# =========================
final_hybrid_df = doc_df.alias("d") \
    .join(
        profit_df.alias("p"),
        on=["company"],
        how="inner"
    ) \
     .join(
        balance_df.alias("b"),
        on=["company"],
        how="inner"
    ) \
    .join(
        employee_df.alias("e"),
        on=["company"],
        how="inner"
    )

# =========================
# JOIN WITH DOCUMENT TABLE
# =========================


    

# =========================
# DISPLAY
# =========================

display(final_hybrid_df)

# =========================
# SAVE FINAL TABLE
# =========================

final_hybrid_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "enterprise_ai.raw_data.final_hybrid_rag"
    )

print("final_hybrid_rag table created")

In [0]:
from pyspark.sql import functions as F

# 1. Define your source table (replace with your actual table name)
source_table = "enterprise_ai.raw_data.final_hybrid_rag"
df = spark.table(source_table)

# 2. Create the 'embedding_content' column 
# We combine the text-heavy columns so the Vector Model understands the full context
df_prepared = df.withColumn("id", F.monotonically_increasing_id()) \
    .withColumn("embedding_content", F.concat_ws(" | ", 
        F.col("product_doc"),
        F.lit("Profit Details:"), F.col("profit_particulars"),
        F.lit("Employee Details:"), F.col("employee_particulars")
    ))

# 3. Select final schema
# Filterable columns (Indexed) and Content columns (Vectorized)
df_final = df_prepared.select(
    "id",
    "company",
    "financial_year",
    "product_name",
    "embedding_content",
    # We keep the numeric columns here for SQL-based retrieval later
    "profit_fy_26", "profit_fy_25", "profit_fy_24",
    "balance_fy_26", "balance_fy_25", "balance_fy_24",
    "employee_fy_26", "employee_fy_25", "employee_fy_24"
)

# 4. Write to a new Delta Table
# This is the table you will point your Vector Search Index to
table_name = "enterprise_ai.raw_data.final_hybrid_rag_embeddings"
df_final.write.format("delta").mode("overwrite").saveAsTable(table_name)

print(f"Table {table_name} created successfully.")

In [0]:
spark.sql("""
ALTER TABLE enterprise_ai.raw_data.final_hybrid_rag_embeddings
SET TBLPROPERTIES (delta.enableChangeDataFeed = true )
          """)
